<a href="https://colab.research.google.com/github/hmsquare/DHmethods2/blob/main/261021_dh2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

研究テーマ：独学者向けの機械翻訳システムの開発

構想：WikipediaなどのWebページを機械翻訳で独学者のレベルに合わせて学習用の文章を生成する

（例：英語からやさしい日本語に翻訳する、が、「やさしい」の定義が未定）

現時点の考え：Wikipediaからデータを取り、原文の難しさがいかに翻訳結果に影響があるのかを検証

（英語版のデータ量が膨大すぎるため、こちらの課題ではやさしい英語版のものだけ扱います）

# Setup

Initialize output location

In [ ]:
from pathlib import Path

OUT = Path("_out")
OUT.mkdir(exist_ok=True)

Check /content of session

In [ ]:
!ls -lah /content

total 20K
drwxr-xr-x 1 root root 4.0K Oct 28 08:02 .
drwxr-xr-x 1 root root 4.0K Oct 28 07:42 ..
drwxr-xr-x 4 root root 4.0K Oct 26 22:18 .config
drwxr-xr-x 2 root root 4.0K Oct 28 08:02 _out
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


# 261021 構造化データの処理
HTML、XMLからの情報抽出を、授業で扱ったものをアレンジするか、自身の研究テーマに置き換えて実行した結果を共有してください。

## Get data

Sources

In [ ]:
orig = 'enwiki-20261001-pages-articles-multistream.xml.bz2'
simp = 'simplewiki-20261001-pages-articles-multistream.xml.bz2'

In [ ]:
# Constructing the corresponding index filenames
orig_index = orig.replace('.xml.bz2', '-index.txt.bz2')
simp_index = simp.replace('.xml.bz2', '-index.txt.bz2')

print(f"Original Dump: {orig}")
print(f"Original Index: {orig_index}\n")
print(f"Simple Dump: {simp}")
print(f"Simple Index: {simp_index}")

Original Dump: enwiki-20261001-pages-articles-multistream.xml.bz2
Original Index: enwiki-20261001-pages-articles-multistream-index.txt.bz2

Simple Dump: simplewiki-20261001-pages-articles-multistream.xml.bz2
Simple Index: simplewiki-20261001-pages-articles-multistream-index.txt.bz2


Download dumps

In [ ]:
# simp xml
!wget -P {OUT} https://dumps.wikimedia.org/simplewiki/20261001/simplewiki-20261001-pages-articles-multistream.xml.bz2

--2026-10-28 08:03:02--  https://dumps.wikimedia.org/simplewiki/20261001/simplewiki-20261001-pages-articles-multistream.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.242, 2620:0:861:ed1a::3:242
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.242|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 387919857 (370M) [application/octet-stream]
Saving to: ‘_out/simplewiki-20261001-pages-articles-multistream.xml.bz2’

simplewiki-20261001 100%[===================>] 369.95M  4.91MB/s    in 76s     

2026-10-28 08:04:18 (4.89 MB/s) - ‘_out/simplewiki-20261001-pages-articles-multistream.xml.bz2’ saved [387919857/387919857]



In [ ]:
# simp index
!wget -P {OUT} https://dumps.wikimedia.org/simplewiki/20261001/simplewiki-20261001-pages-articles-multistream-index.txt.bz2

--2026-10-28 08:04:18--  https://dumps.wikimedia.org/simplewiki/20261001/simplewiki-20261001-pages-articles-multistream-index.txt.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.242, 2620:0:861:ed1a::3:242
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.242|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5251971 (5.0M) [application/octet-stream]
Saving to: ‘_out/simplewiki-20261001-pages-articles-multistream-index.txt.bz2’

simplewiki-20261001 100%[===================>]   5.01M  5.14MB/s    in 1.0s    

2026-10-28 08:04:19 (5.14 MB/s) - ‘_out/simplewiki-20261001-pages-articles-multistream-index.txt.bz2’ saved [5251971/5251971]



In [ ]:
root = '/content/_out/'
index = root + 'simplewiki-20261001-pages-articles-multistream-index.txt.bz2'
pages = root + 'simplewiki-20261001-pages-articles-multistream.xml.bz2'

Unzip data

In [ ]:
# Option 1: Using shell commands (fast and easy in Colab)
# This decompresses the file and removes the original .bz2 file.
# !bzip2 -d /content/_out/simplewiki-20261001-pages-articles-multistream-index.txt.bz2

# If you want to keep the original .bz2 file, use the -k flag:
# !bzip2 -dk /content/_out/simplewiki-20261001-pages-articles-multistream-index.txt.bz2

# Option 2: Using Python's built-in bz2 module
import bz2

def unzipbz(infn, outfn):
  try:
    with bz2.open(infn, "rb") as f_in:
        with open(outfn, "wb") as f_out:
            f_out.write(f_in.read())
    print(f"Successfully decompressed to: {outfn}")
  except FileNotFoundError:
      print(f"File not found: {infn}. Please make sure the file is downloaded.")

In [ ]:
intxt = root + 'index.txt'
unzipbz(index, intxt)

Successfully decompressed to: /content/_out/index.txt


In [ ]:
pgxml = root + 'pages.xml'
unzipbz(pages, pgxml)

Successfully decompressed to: /content/_out/pages.xml


## Exploring the downloaded files


Index .txt file

In [ ]:
# Read and print the first 10 lines of the index file
with open(intxt, 'r', encoding='utf-8') as f:
    for i in range(10):
        line = f.readline()
        if not line:
            break
        print(line.strip())

552:1:April
552:2:August
552:6:Art
552:8:A
552:9:Air
552:11:Wikipedia:Administrators
552:12:Autonomous communities of Spain
552:13:Alan Turing
552:14:Alanis Morissette
552:17:Adobe Illustrator


Pages .xml file

In [ ]:
# Open and print the first x lines of the raw XML file
with open(pgxml, 'r', encoding='utf-8') as f:
    for i in range(30):
        line = f.readline()
        if not line:
            break
        print(line, end='')

<mediawiki xmlns="http://www.mediawiki.org/xml/export-0.11/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.mediawiki.org/xml/export-0.11/ http://www.mediawiki.org/xml/export-0.11.xsd" version="0.11" xml:lang="en">
  <siteinfo>
    <sitename>Wikipedia</sitename>
    <dbname>simplewiki</dbname>
    <base>https://simple.wikipedia.org/wiki/Main_Page</base>
    <generator>MediaWiki 1.47.0-wmf.21</generator>
    <case>first-letter</case>
    <namespaces>
      <namespace key="-2" case="first-letter">Media</namespace>
      <namespace key="-1" case="first-letter">Special</namespace>
      <namespace key="0" case="first-letter" />
      <namespace key="1" case="first-letter">Talk</namespace>
      <namespace key="2" case="first-letter">User</namespace>
      <namespace key="3" case="first-letter">User talk</namespace>
      <namespace key="4" case="first-letter">Wikipedia</namespace>
      <namespace key="5" case="first-letter">Wikipedia talk</namespace>


すでにある程度整理されていたデータがあるため、こちらはスクレイピングせずにします

In [ ]:
import xml.etree.ElementTree as ET